# Data Quality Correction

Before performing EDA (Exploratory Data Analysis), some initial data cleaning is necessary so you can trust what you’re analyzing.

**Basic cleaning includes:**

- Loading the data correctly (e.g., proper parsing of dates, encoding, etc.)

- Checking for and handling:

    - Missing values (at least identifying them)

    - Incorrect data types (e.g., string instead of numeric)

    - Obvious outliers or garbage values (e.g., negative age)

    - Duplicate rows

> 🧼 This stage is often called "initial or light cleaning" — it helps avoid misleading results during EDA.

**See Run Results :** [https://dagshub.com/Rahul-404/heart-stroke-prediction.mlflow](https://dagshub.com/Rahul-404/heart-stroke-prediction.mlflow/#/)

## Required Installtion

In [ ]:
# TRIGGER_FLAG: run
import os
import sys

# Check if running in Kaggle Cloud Environment
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    print("--- Kaggle Environment Detected: Installing Cloud Dependencies ---")
    # 1. Install general cloud dependencies
    os.system(f"{sys.executable} -m pip install -q mlflow dagshub")
    
    # 2. Clone the specific branch of the repository
    print("Cloning repository...")
    os.system("git clone -b feat/notebooks https://github.com/Rahul-Shelke-1/heart-stroke-risk-stratification.git")
    
    # 3. Change directory into the cloned repo and install it in editable mode
    if os.path.exists("heart-stroke-risk-stratification"):
        os.chdir("heart-stroke-risk-stratification")
        print("Installing package in editable mode...")
        os.system(f"{sys.executable} -m pip install -e .")
        # Optional: Change directory back to the original root if needed
        # os.chdir("..")
    else:
        print("Error: Repository directory not found.")
        
else:
    print("--- Local/Non-Kaggle Environment Detected: Skipping Installation ---")
    # Local packages should ideally be pre-managed via your local uv environment

## Smart Environment Detection & Secrets Setup

In [ ]:
import os
import mlflow
from dotenv import load_dotenv

# 1. Safely retrieve credentials from Kaggle's internal secure vault
try:
    # 1. Detect if running inside Kaggle's container
    IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

    if IS_KAGGLE:
        DAGSHUB_USERNAME = os.environ.get("DAGSHUB_USERNAME")
        DAGSHUB_TOKEN = os.environ.get("DAGSHUB_TOKEN")
    else:
        load_dotenv()
        DAGSHUB_USERNAME = os.environ.get("DAGSHUB_USERNAME", "Rahul-404")
        DAGSHUB_TOKEN = os.environ.get("DAGSHUB_TOKEN")
    
    # Replace this with your actual DagsHub repo name
    REPO_NAME = "heart-stroke-prediction"
    
    # 2. Inject environment variables that the MLflow client natively looks for
    os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USERNAME
    os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN
    
    # 3. Set the remote tracking URI to point to DagsHub
    tracking_uri = f"https://dagshub.com/{DAGSHUB_USERNAME}/{REPO_NAME}.mlflow"
    mlflow.set_tracking_uri(tracking_uri)
    
    print("Successfully connected to DagsHub MLflow tracking server!")
except Exception as e:
    # print(f"Local or non-Kaggle execution environment detected: {e}")
    print(f"Error establishing MLflow/DagsHub context: {e}")

## Unified Path Setup Strategy

In [ ]:
import os
from pathlib import Path

# 1. Reuse your environment checker flag
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

# 2. Define your repository name to map Kaggle input paths perfectly
KAGGLE_USER = os.environ.get("KAGGLE_USER")  # Replace with your actual lowercase Kaggle username
DATASET_SLUG = os.environ.get("KAGGLE_DATASET_NAME")  # Replace with your Kaggle dataset slug

if IS_KAGGLE:
    # --- KAGGLE CLOUD PATH CONTEXT ---
    # Kaggle mounts your datasets under /kaggle/input/YOUR_DATASET_SLUG
    INPUT_DIR = Path(f"/kaggle/input/datasets/{KAGGLE_USER}/{DATASET_SLUG}")

    if not INPUT_DIR.exists():
        raise f"Input Directory not sxists: {INPUT_DIR}"
    
    # Kaggle strictly allows file writing ONLY inside /kaggle/working
    OUTPUT_DIR = Path("/kaggle/working/artifacts")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    if not OUTPUT_DIR.exists():
        raise f"Output Directory not sxists: {OUTPUT_DIR}"
else:
    # --- LOCAL ENVIRONMENT PATH CONTEXT ---
    current_directory = Path.cwd()
    # Points to your local repository structure (e.g., repository_root/data/)
    INPUT_DIR = Path(current_directory, "notebooks", "data")
    
    if not INPUT_DIR.exists():
        raise f"Input Directory not sxists: {INPUT_DIR}"

    # Keeps your local repo clean by saving local run artifacts into an outputs folder
    OUTPUT_DIR = Path(current_directory, "notebooks", "artifact")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("--- PATH SUMMARY ---")
print(f"Execution Target: {'Kaggle Cloud' if IS_KAGGLE else 'Local Machine'}")
print(f"Reading Input From: {INPUT_DIR}")
print(f"Writing Outputs To: {OUTPUT_DIR}")

## Importing Libraries

In [ ]:
from src.analyze.data_correction.schema_correction import (MissingValueNormalizer, FrequencyFilter)
from src.analyze.data_correction.schema_correction import (ColumnNameCorrector, ValueCorrector)
from src.analyze.data_inspection.inspector import InspectData
from sklearn.pipeline import Pipeline
import mlflow.sklearn
import pandas as pd
import cloudpickle
import warnings
import mlflow
import os

warnings.filterwarnings("ignore")

In [ ]:
DATA_FILES = os.listdir(INPUT_DIR)
DATA_FILES

In [ ]:
ARTIFACT_FILES = os.listdir(OUTPUT_DIR)
ARTIFACT_FILES

In [ ]:
DATA_FILE_NAME = {
    'raw': Path(INPUT_DIR, 'healthcare-dataset-stroke-data.csv'),
    'clean': Path(OUTPUT_DIR, 'heart_stroke_data.csv'),
}

ARTIFACT_FILE_NAME = {
    'data_correct': Path(OUTPUT_DIR, 'data_correction_pipeline.pkl'),
}

### Set Experiment Name

In [ ]:
experiment_name = "01_Data_Quality_Correction"
experiment_tags = {
    "experiment_type": "Classification",
    "project": "heart_stroke_risk_stratification",
    "team": "Data_Science_Core"
}

try:
    experiment_id = mlflow.create_experiment(name=experiment_name, tags=experiment_tags)
except Exception:
    # If it already exists, fetch its ID
    experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

# 3. Set it as the active experiment
mlflow.set_experiment(experiment_id=experiment_id)

### Set Run Name

In [ ]:
# Starts a global active run
mlflow.start_run(run_name="Data_Quality_Correction_Session")

## Import Data

In [ ]:
df = pd.read_csv(DATA_FILE_NAME['raw'])
dataset = mlflow.data.from_pandas(df, name="raw_data", targets="stroke")
mlflow.log_input(dataset, context="dataset_ingestion")
mlflow.log_param("raw_data_path",DATA_FILE_NAME['raw'])

In [ ]:
mlflow.log_param("num_train_samples", df.shape[0])
mlflow.log_param("num_features", df.shape[1])
df.shape

In [ ]:
df.head()

In [ ]:
df['stroke'].value_counts()

In [ ]:
stroke_proportions = df['stroke'].value_counts(normalize=True).to_dict()

mlflow.log_param("target_class_0_prop", stroke_proportions[0])
mlflow.log_param("target_class_1_prop", stroke_proportions[1])

stroke_proportions

In [ ]:
df['stroke'].value_counts(normalize=True)

## 1. Check Schema, shape, columns

In [ ]:
df.info()

- **~ 5k rows**
- **12 columns** (including target column)

## 2. Standardize Column Names

In [ ]:
df.columns

- will keep all to lower case.

In [ ]:
column_correction_dict = {
    'Residence_type': 'residence_type'
}

# Save the dictionary directly to MLflow as a JSON file
mlflow.log_dict(column_correction_dict, "artifact/preprocessing/column_mapping.json")

In [ ]:
column_corrector = ColumnNameCorrector(column_correction_dict)

In [ ]:
df = column_corrector.transform(df)

In [ ]:
df.head()

## 3. Fix Categorical Values (String Standardization)

In [ ]:
inspector = InspectData(df)

In [ ]:
df.columns

In [ ]:
cat_col = ['gender', 'hypertension', 'heart_disease', 'ever_married',
       'work_type', 'residence_type', 'smoking_status', 'stroke']

In [ ]:
inspector.categorical_column_inspect(cat_col)

**Correction needed to this columns:**
- gender
- ever_married
- work_type
- residence_type
- smoking_status

In [ ]:
value_correction_dict = {
    'gender': {'Male': 'male', 'Female': 'female', 'Other': 'other'},
    'ever_married': {'Yes':'yes', 'No': 'no'},
    'residence_type': {'Urban': 'urban', 'Rural': 'rural'},
    'smoking_status': {'formerly smoked': 'formerly_smoked', 
                            'never smoked': 'never_smoked', 
                            'Unknown': 'unknown'},
    'work_type': {'Private':'private', 'self-employed': 'self_employed', 
                            'Govt_job': 'govt_job', 'Never_worked': 'never_worked',
                            'children': 'children'},
}

mlflow.log_dict(value_correction_dict, "artifact/preprocessing/value_mapping.json")

In [ ]:
value_corrector = ValueCorrector(column_mappings=value_correction_dict)

In [ ]:
df = value_corrector.transform(X=df)

In [ ]:
df.head()

In [ ]:
df.tail()

## 4. Fix Categorical Values Frequency (Handling category frequency)

In [ ]:
frequency_filter = FrequencyFilter(threshold=1, action='drop')

# Log the initialization parameters
mlflow.log_param("freq_filter_threshold", frequency_filter.threshold)
mlflow.log_param("freq_filter_action", frequency_filter.action)

In [ ]:
df = frequency_filter.fit_transform(df)

In [ ]:
df.shape

In [ ]:
df[df['gender'] == 'other']

In [ ]:
df['gender'].value_counts()

In [ ]:
# Assuming your class stores the dropped labels in an attribute after fitting
if hasattr(frequency_filter, "fill_value"):
    mlflow.log_dict(
        {"dropped_labels": [frequency_filter.fill_value]},
        "artifact/preprocessing/dropped_rare_categories.json",
    )

## 5. Fix Numerical Values (Context Standardization)

In [ ]:
num_col = ['age', 'avg_glucose_level', 'bmi']

In [ ]:
inspector.numerical_column_inspect(num_col)

**bmi**

- `bmi` feature has 4% (201) values missing
<!-- - replacing `nan` with -1 -->

⚠️ **Unusual / Suspicious Values:**

| Range                               | Possible Issue                                                   |
| ----------------------------------- | ---------------------------------------------------------------- |
| **< 40**                            | Rare, possible hypoglycemia or data error                        |
| **> 250–300**                       | Very high, may indicate poorly controlled diabetes or unit error |
| **> 600**                           | Extremely rare, possibly a data or entry error                   |
| **Unrealistic values (e.g. >1000)** | Almost certainly data error                                      |


## 6. Remove Unwanted/Corrupted Columns/Rows

In [ ]:
df[df.duplicated()]

In [ ]:
df[df.drop(columns="id", axis=1).duplicated()]

## 7. Identifying missing values (off context)

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df[df['smoking_status'] == 'unknown']

In [ ]:
missing_normalizer = MissingValueNormalizer(missing_tokens=['unknown'])

In [ ]:
df = missing_normalizer.fit_transform(df)

In [ ]:
df.isna().sum()

In [ ]:
missing_series = df.isnull().sum()
missing_params = {f"post_norm_missing_{col}": int(val) for col, val in missing_series.items()}

In [ ]:
# Log the normalizer configurations
mlflow.log_param("missing_normalizer_tokens", str(missing_normalizer.missing_tokens))

# Log all column missing counts simultaneously using a single dictionary batch
mlflow.log_params(missing_params)

## **Pipeline: Basic Cleaning**

### 1. Build pipeline

In [ ]:
# Create pipeline
data_correction_pipeline = Pipeline(
    steps=[
        ('correct_column_names', ColumnNameCorrector(mapping=column_correction_dict)),
        ('correct_column_values', ValueCorrector(column_mappings=value_correction_dict)),
        ('missing_normalizer', MissingValueNormalizer(missing_tokens=['unknown'])),
        ('frequency_filter', FrequencyFilter(threshold=1, action='drop')),
    ]
)

In [ ]:
data_correction_pipeline

#### data-head

In [ ]:
df.head()

In [ ]:
data_correction_pipeline.fit_transform(df).head()

#### data-tail

In [ ]:
df.tail()

In [ ]:
data_correction_pipeline.fit_transform(df).tail()

### 2. Apply pipeline

In [ ]:
df = pd.read_csv(DATA_FILE_NAME['raw'])

# df.head()

In [ ]:
df.isna().sum()

In [ ]:
df_corrected = data_correction_pipeline.fit_transform(df)

In [ ]:
df_corrected.head()

In [ ]:
df_corrected.tail()

In [ ]:
df_corrected.isna().sum()

### 3. Save pipeline & Data

In [ ]:
df_corrected.to_csv(DATA_FILE_NAME['clean'], index=False)

In [ ]:
local_file_path = ARTIFACT_FILE_NAME["data_correct"]

with open(local_file_path, "wb" ) as f:
    cloudpickle.dump(data_correction_pipeline, f)

In [ ]:
with open(local_file_path, "rb" ) as f:
    data_correction_pipeline = cloudpickle.load(f)

In [ ]:
data_correction_pipeline

In [ ]:
mlflow.log_artifact(local_file_path, artifact_path="artifact/pipelines/data_cleaning")

# Optional: Log a metadata tag identifying it as a static pipeline
mlflow.set_tag("pipeline.type", "static_etl")

In [ ]:
# Make sure to close the run so MLflow knows the notebook is done
mlflow.end_run()

### **Next Action:**

- Handle Missing Values
- Handle Outliers